# Faithfulness of CoT Traces in Small Medical QA Models - Full Pipeline

Combined notebook (phases 00-05 in order). Run top to bottom on a Colab GPU runtime.
Upload `mini_project.zip` to Colab and `!unzip -q mini_project.zip -d /content/` first; the first setup cell verifies this. After each phase, run that phase's download cell and share `outputs.zip`. Phase 3 (taxonomy) requires human annotation - see its markdown cell.


---

# Phase 0 - Setup & smoke test

*(from `00_setup_and_smoke.ipynb`)*

# 00 - Setup & smoke test\n\nVerifies GPU, installs deps, and runs a 3-question smoke generation with the primary model. **Expected:** 3 traces with parsed answers.

**▶ Step 0 - Environment & project setup (install deps, GPU check, HF token)**

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


**▶ Step 1 - Model vetting: smoke-test generation with the primary model**

In [ ]:
from src.common import load_config, load_prompt_config, load_model_and_tokenizer, set_seed
from src.inference import build_prompt, generate_and_score, parse_answer_letter, letter_token_ids
cfg = load_config()
pcfg = load_prompt_config()
set_seed(cfg["seed"])
model_id = cfg["model"]["primary"]
print("model:", model_id)
model, tok = load_model_and_tokenizer(model_id, dtype=cfg["model"]["dtype"])
# tiny smoke set: first 3 MedQA questions
from src.data_loading import load_medqa
items = load_medqa(n=3, seed=cfg["seed"])
for it in items:
    rec = generate_and_score(model, tok, pcfg, it, max_new_tokens=256)
    print("="*80)
    print("Q:", it["question"][:200])
    print("gold:", it["gold"], "| predicted:", rec["predicted"], "| P_clean:", rec["P_clean"])
    print("trace tail:", rec["trace"][-300:])



---

# Phase 1 - Behavioral faithfulness (Week 1)

*(from `01_phase1_behavioral.ipynb`)*

# 01 - Phase 1: behavioral faithfulness (Week 1)\n\nSteps 2-5: sample 400 MedQA questions, generate CoT traces, build evidence/control perturbations, score them, compute F(x) + flip rates + Wilcoxon, and select the 100-question mechanistic subset.\n\n**Runtime on a T4 GPU:** a few hours for 400 questions x 3 runs. Results checkpoint after every question, so you can stop/resume safely (re-running skips finished ids).

**▶ Step 0 - Environment & project setup (install deps, GPU check, HF token)**

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


**▶ Steps 2-5 - Phase 1: MedQA sample -> CoT generation (Step 3) -> evidence/control perturbations (Step 4) -> F(x) + flip rates + Wilcoxon (Step 5) -> 100-question subset**

In [ ]:
from src import pipeline
from src.common import load_config
cfg = load_config()
# Override the model here if you want a different candidate:
# cfg["model"]["primary"] = "akshayballal/Qwen3-1.7B-Pubmed-16bit-GRPO"
stats = pipeline.run_phase1(cfg)
print("PHASE 1 DONE")
import json; print(json.dumps(stats, indent=2))


**▶ Download outputs.zip (share with me after Phase 1)**

In [ ]:
# @title Download outputs (run after the phase finishes)
import os, zipfile, glob
from google.colab import files
out_dir = "outputs"
zip_path = "/content/outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(os.path.join(out_dir, "**", "*"), recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, "/content"))
files.download(zip_path)
print("Downloaded", zip_path)



---

# Phase 2a - Logit lens (Week 2)

*(from `02_phase2_logit_lens.ipynb`)*

# 02 - Phase 2a: logit lens on the 100-question subset (Week 2)\n\nStep 7: per-layer probability of each option letter at the answer position; emergence layer + earliness score. Requires Phase 1 outputs (run notebook 01 first, or place its outputs in `outputs/`).

**▶ Step 0 - Environment & project setup**

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


**▶ Step 7 - Phase 2a: logit lens on the 100-question subset (emergence layer, earliness)**

In [ ]:
from src import pipeline
from src.common import load_config
cfg = load_config()
recs = pipeline.run_phase2_logit_lens(cfg)
ok = [r for r in recs if "earliness" in r]
print(f"logit lens done for {len(ok)} questions")
if ok:
    import statistics
    print("mean emergence layer:", statistics.mean(r["emergence_layer"] for r in ok))
    print("mean earliness:", statistics.mean(r["earliness"] for r in ok))


**▶ Download outputs.zip**

In [ ]:
# @title Download outputs (run after the phase finishes)
import os, zipfile, glob
from google.colab import files
out_dir = "outputs"
zip_path = "/content/outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(os.path.join(out_dir, "**", "*"), recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, "/content"))
files.download(zip_path)
print("Downloaded", zip_path)



---

# Phase 2b - Activation patching + agreement (Week 3)

*(from `03_phase2_patching_and_agreement.ipynb`)*

# 03 - Phase 2b: activation patching + behavioral/mechanistic agreement (Week 3)\n\nStep 9: patch clean activations into the corrupted run at each (layer, position) of the redacted span; recovery score on the correct option's logit. Step 10: Spearman rho + Cohen's kappa between F(x) and mechanistic scores; off-diagonal cases are routed to the Phase 3 review.

**▶ Step 0 - Environment & project setup**

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


**▶ Steps 9-10 - Phase 2b: activation patching (Step 9) + behavioral/mechanistic agreement: Spearman rho, Cohen's kappa, off-diagonal routing (Step 10)**

In [ ]:
from src import pipeline
from src.common import load_config
cfg = load_config()
recs = pipeline.run_phase2_patching(cfg)
ok = [r for r in recs if r.get("best")]
print(f"patching done for {len(ok)} questions")
stats, case_ids = pipeline.run_agreement(cfg)
import json; print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "off_diagonal_ids"}
      for k, v in stats.items()}, indent=2))
print("review case ids:", len(case_ids))


**▶ Download outputs.zip**

In [ ]:
# @title Download outputs (run after the phase finishes)
import os, zipfile, glob
from google.colab import files
out_dir = "outputs"
zip_path = "/content/outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(os.path.join(out_dir, "**", "*"), recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, "/content"))
files.download(zip_path)
print("Downloaded", zip_path)



---

# Phase 3 - Error taxonomy annotation (needs human input)

*(from `04_phase3_taxonomy.ipynb`)*

# 04 - Phase 3: error taxonomy annotation (Weeks 2-3, NEEDS HUMAN INPUT)\n\nStep 8/11. Run the first cell to export `outputs/annotation_sheet_50.csv` (N=50 review cases: disagreement cases + incorrect/low-F).\n\n**Human task:** BOTH annotators independently fill the `primary_label` and optional `secondary_label` columns (categories: knowledge_gap, reasoning_slip, hallucinated_premise, question_misreading, confident_unjustified_leap, other) and save as `labels_annotator1.csv` and `labels_annotator2.csv`. Upload both back into `outputs/` and run the last cell to compute agreement (raw % + Cohen's kappa) and the final adjudicated labels.

**▶ Step 0 - Environment & project setup**

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


**▶ Step 8 - Phase 3: export annotation_sheet_50.csv (N=50 review cases) - HUMAN ANNOTATION FOLLOWS**

In [ ]:
import json
from src.common import load_config, read_jsonl, read_records, save_json
from src.taxonomy import export_annotation_sheet, select_review_cases
cfg = load_config()
n = cfg["dataset"]["n_sample"]
data_dir = cfg["paths"]["data_dir"]
out_dir = "outputs"
scored = read_records(f"{out_dir}/scored_{n}.json")
items = read_jsonl(f"{data_dir}/subset_100.jsonl")
traces = read_jsonl(f"{out_dir}/traces_{n}.jsonl")
# full 400-item pool for review cases
all_items = read_jsonl(f"{data_dir}/sample_{n}.jsonl")
off = json.load(open(f"{out_dir}/off_diagonal_ids.json")) if __import__("os").path.exists(f"{out_dir}/off_diagonal_ids.json") else []
case_ids = select_review_cases(scored, off, n_total=50, traces=traces)
save_json(case_ids, f"{out_dir}/review_case_ids.json")
csv_path = export_annotation_sheet(case_ids, all_items, traces, scored,
                                   f"{out_dir}/annotation_sheet_50.csv")
print("Wrote", csv_path, "with", len(case_ids), "cases - download it and annotate.")


**▶ Step 11 - Phase 3: inter-annotator agreement (raw % + Cohen's kappa) + adjudicated final labels - needs the two label CSVs uploaded**

In [ ]:
# After uploading labels_annotator1.csv and labels_annotator2.csv into outputs/:
from src.taxonomy import load_labels, compute_agreement, finalize_labels, taxonomy_summary
l1 = load_labels("outputs/labels_annotator1.csv")
l2 = load_labels("outputs/labels_annotator2.csv")
agree = compute_agreement(l1, l2)
print("Inter-annotator agreement:", agree)
# Adjudication: fill this dict with your discussed decision for each disputed id
adjudicated = {cid: "" for cid in agree["disputed_ids"]}  # <-- HUMAN INPUT
final_csv = finalize_labels(l1, l2, adjudicated, "outputs/taxonomy_labels_final.csv")
summary = taxonomy_summary(final_csv, out_json="outputs/taxonomy_summary.json")
print("Taxonomy summary:", summary)


**▶ Download outputs.zip**

In [ ]:
# @title Download outputs (run after the phase finishes)
import os, zipfile, glob
from google.colab import files
out_dir = "outputs"
zip_path = "/content/outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(os.path.join(out_dir, "**", "*"), recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, "/content"))
files.download(zip_path)
print("Downloaded", zip_path)



---

# Phase 4 - Synthesis & report (Week 4)

*(from `05_phase4_report.ipynb`)*

# 05 - Phase 4: synthesis & report (Week 4)\n\nStep 12: figures + `outputs/report.md` from all phase outputs. Also zips everything for download.

**▶ Step 0 - Environment & project setup**

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


**▶ Step 12 - Phase 4: consolidate results, figures, report.md (Steps 12-14 deliverables)**

In [ ]:
from src import pipeline
from src.common import load_config
cfg = load_config()
res = pipeline.run_report(cfg)
print("figures:", res["figures"])
print("report:", res["report"])
print(open(res["report"]).read()[:2000])


**▶ Download outputs.zip (final)**

In [ ]:
# @title Download outputs (run after the phase finishes)
import os, zipfile, glob
from google.colab import files
out_dir = "outputs"
zip_path = "/content/outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(os.path.join(out_dir, "**", "*"), recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, "/content"))
files.download(zip_path)
print("Downloaded", zip_path)
